# Gold-Silver Pairs Trading — Full Analysis

Single notebook spanning all phases of `PLAN.md`. Each phase gets its own section so the evolution of the strategy is visible end to end.

## Sections
1. Data exploration
2. Cointegration analysis
3. Spread construction
4. OU modeling and half-life
5. Signal generation
6. Backtest (in-sample → walk-forward)
7. Performance analytics
8. Stress testing and regime analysis

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Data Exploration *(Phase 1)*

Pull GLD / SLV adjusted-close prices through yfinance. The cleaning pass uses the yfinance trading-day index as canonical (no synthetic business-day reindex — that would invent observations on NYSE holidays). Multi-day market closures (Hurricane Sandy, COVID-era halts) are logged as anomalies but never imputed.

In [ ]:
from src.config import load_config
from src.data_loader import load_pair_data

cfg = load_config(ROOT / "config.yaml")
df = load_pair_data(cfg, refresh=False)
print(f"rows       : {len(df):,}")
print(f"date range : {df.index.min().date()} -> {df.index.max().date()}")
print(f"NaNs       : {int(df.isna().sum().sum())}")
df.head()

In [ ]:
df[["gold","silver"]].describe().round(2)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.plot(df.index, df["gold"], color="#c9a227", label="GLD")
ax2.plot(df.index, df["silver"], color="#9aa0a6", label="SLV")
ax1.set_ylabel("GLD ($)", color="#c9a227")
ax2.set_ylabel("SLV ($)", color="#9aa0a6")
ax1.set_title("GLD & SLV adjusted close")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ret = df[["log_gold","log_silver"]].diff().dropna()
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0,0].plot(ret.index, ret["log_gold"],   lw=0.5, color="#c9a227"); axes[0,0].set_title("GLD daily log return"); axes[0,0].grid(alpha=0.3)
axes[0,1].plot(ret.index, ret["log_silver"], lw=0.5, color="#9aa0a6"); axes[0,1].set_title("SLV daily log return"); axes[0,1].grid(alpha=0.3)
axes[1,0].hist(ret["log_gold"],   bins=80, color="#c9a227", alpha=0.85); axes[1,0].set_title("GLD log-return distribution"); axes[1,0].axvline(0, color='k', lw=0.5)
axes[1,1].hist(ret["log_silver"], bins=80, color="#9aa0a6", alpha=0.85); axes[1,1].set_title("SLV log-return distribution"); axes[1,1].axvline(0, color='k', lw=0.5)
plt.tight_layout()
plt.show()

print(f"GLD/SLV daily-return correlation: {ret['log_gold'].corr(ret['log_silver']):.3f}")

## 2. Cointegration Analysis *(Phase 2)*

Three layers of evidence:

1. **ADF on each leg** — expect to fail to reject the unit-root null. Confirms each series is I(1).
2. **Engle-Granger two-step** — regress `log(GLD) = α + β·log(SLV) + ε`, ADF on residuals. Rejecting null means cointegrated.
3. **Johansen trace test** — secondary check, also yields the cointegrating vector for cross-validation against EG.

Then **rolling Engle-Granger** on 504-day (≈2yr) sliding windows with 21-day step. Plan predicts visible p-value spikes near 2011 silver squeeze and 2020 COVID — flat plot would mean a bug.

In [ ]:
from src.cointegration import adf_test, engle_granger_test, johansen_test, rolling_cointegration

adf_g = adf_test(df['log_gold'])
adf_s = adf_test(df['log_silver'])
print('--- ADF on each leg (null: unit root) ---')
print(f'log(GLD): stat={adf_g.statistic:.3f}, p={adf_g.pvalue:.4f}, crit5%={adf_g.crit_5pct:.3f}')
print(f'log(SLV): stat={adf_s.statistic:.3f}, p={adf_s.pvalue:.4f}, crit5%={adf_s.crit_5pct:.3f}')
print('-> both p > 0.05: each leg is I(1) as required.' if adf_g.pvalue > 0.05 and adf_s.pvalue > 0.05 else '-> WARNING: one leg appears stationary; cointegration framework may not apply.')

In [ ]:
eg = engle_granger_test(df['log_gold'], df['log_silver'])
print('--- Engle-Granger: log(GLD) = α + β·log(SLV) + ε ---')
print(f'fitted: β={eg.beta:.4f}, α={eg.alpha:.4f}, n={eg.n_obs}')
print(f'residual ADF stat = {eg.residual_adf.statistic:.3f}')
print(f'crit values (1/5/10%): {eg.residual_adf.crit_1pct:.3f} / {eg.residual_adf.crit_5pct:.3f} / {eg.residual_adf.crit_10pct:.3f}')
print(f'MacKinnon p-value: {eg.pvalue:.4f}')
if eg.pvalue < 0.05:
    print('-> reject unit root in residuals: full-sample cointegration.')
else:
    print('-> FAIL to reject unit root in residuals: full-sample NOT cointegrated.')

In [ ]:
jh = johansen_test(df['log_gold'], df['log_silver'])
print('--- Johansen trace test ---')
print(f'trace stats   r=0: {jh.trace_stats[0]:.3f},  r<=1: {jh.trace_stats[1]:.3f}')
print(f'95% crits     r=0: {jh.crit_values_95[0]:.3f},  r<=1: {jh.crit_values_95[1]:.3f}')
print(f'cointegrating vector (normalized): ({jh.cointegrating_vector[0]:.4f}, {jh.cointegrating_vector[1]:.4f})')
print(f'   implied β from Johansen: {-jh.cointegrating_vector[1]:.4f}')
print(f'   Engle-Granger β (above): {eg.beta:.4f}')
print(f'#cointegrating relations at 95%: {jh.n_cointegrating_relations_at_95}')

### Rolling Engle-Granger

Cached to `data/processed/rolling_coint.parquet` — the next ~200 windows would otherwise re-run an ADF on every slide.

In [ ]:
roll = rolling_cointegration(
    df['log_gold'], df['log_silver'],
    window=cfg.cointegration.rolling_window_days,
    step=cfg.cointegration.rolling_step_days,
    cache_path=cfg.cointegration.rolling_cache_path,
)
print(f'windows: {len(roll)},  span {roll.index.min().date()} -> {roll.index.max().date()}')
print(f'mean p-value             : {roll["pvalue"].mean():.4f}')
print(f'pct windows with p < 0.05: {(roll["pvalue"] < 0.05).mean()*100:.1f}%')
print(f'pct windows with p < 0.10: {(roll["pvalue"] < 0.10).mean()*100:.1f}%')
print(f'rolling β range          : {roll["beta"].min():.3f} -> {roll["beta"].max():.3f} (mean {roll["beta"].mean():.3f})')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True, gridspec_kw={'height_ratios':[2,1]})
ax1.plot(roll.index, roll['pvalue'], color='#2c5fbd', lw=1.4)
ax1.axhline(0.05, color='red',    ls='--', lw=0.8, label='p=0.05')
ax1.axhline(0.10, color='orange', ls='--', lw=0.8, label='p=0.10')
for span_start, span_end, label in [
    ('2008-09-01','2009-03-31','2008 crisis'),
    ('2011-04-01','2011-09-30','2011 silver squeeze'),
    ('2020-02-15','2020-04-30','COVID dislocation'),
    ('2022-01-01','2022-12-31','2022 inflation shock'),
]:
    ax1.axvspan(pd.Timestamp(span_start), pd.Timestamp(span_end), color='gray', alpha=0.15)
    ax1.text(pd.Timestamp(span_start), 0.95, label, fontsize=8, alpha=0.7)
ax1.set_ylabel('Engle-Granger p-value')
ax1.set_title('Rolling cointegration p-value (504-day window, 21-day step)')
ax1.set_ylim(0, 1)
ax1.legend(loc='upper right')
ax1.grid(alpha=0.3)

ax2.plot(roll.index, roll['beta'], color='#444', lw=1.0)
ax2.set_ylabel('rolling β')
ax2.set_xlabel('window end date')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Phase 2 finding

**Full-sample cointegration is rejected** (Engle-Granger p ≈ 0.34, Johansen trace 7.77 vs crit 15.49). But rolling tests show the relationship is **regime-dependent** — cointegrated in roughly 9% of overlapping 2-year windows, with breakdown clustering in known dislocation regimes (2011 silver squeeze, 2019-20 COVID, 2024-26 gold rally). Rolling β wanders from 0.03 to 1.12, ruling out a static hedge ratio.

**Implication for the strategy:** the rolling-p-value filter (originally a Phase 8 mitigation) is now load-bearing. We will only generate signals during windows where the rolling EG p-value is below threshold. Rolling β in spread construction (Phase 3) is mandatory, not a refinement.